# Análisis Exploratorio de Datos

In [ ]:
import pandas as pd
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from category_encoders import TargetEncoder
import pandas as pd
# pip install pyarrow

### Objetivo 1: Predicción del Retorno del Día Siguiente
- Objetivo: entrenar un modelo de machine learning para predecir si una acción aumentará más de un 1% en el siguiente día de trading, utilizando información de los últimos 60 días de datos del mercado del S&P 500 y sus empresas.
- Descripción: el modelo aprovechará indicadores técnicos, acción del precio, dinámica del volumen, medidas de volatilidad y variables rezagadas para aprender patrones de mercado a corto plazo que preceden movimientos positivos significativos del precio. Esta tarea se plantea como un problema de clasificación binaria, donde la variable objetivo indica si el retorno del día siguiente supera el umbral del 1%.
- Propósito: evaluar la viabilidad de la predicción de retornos a corto plazo utilizando datos históricos de mercado, evitando estrictamente el look-ahead bias y preservando la estructura temporal de las series financieras.

### Origen de los datos y descripción
Los datos de este proyecto provienen de Kaggle, del dataset Advanced Stock Dataset creado por baidalinadilzhan, que a su vez se basa en datos descargados de Yahoo Finance API, cubriendo aproximadamente los últimos cinco años de mercado para los componentes del índice S&P 500. Este conjunto de datos incluye observaciones diarias ajustadas por eventos corporativos como dividendos y splits, y está diseñado específicamente para análisis financiero y modelado de series temporales.
En total, el dataset contiene más de 600 000 registros diarios con 73 características que incluyen precios (Open, High, Low, Close), volumen, indicadores técnicos como medias móviles y RSI, métricas de volatilidad y múltiples variables con información histórica rezagada (lags), lo que lo hace adecuado para tareas de predicción de retornos, clasificación direccional y análisis de riesgo en machine learning financiero.

## Columnas del Dataset
- DATE: Fecha del día de trading. 
- TICKER: Símbolo bursátil de la acción.
- OPEN: Precio de apertura del día.
- HIGH: Precio máximo alcanzado durante el día.
- LOW: Precio mínimo alcanzado durante el día.
- CLOSE: Precio de cierre del día.
- VOLUME: Número total de acciones negociadas.
- DIVIDENDS: Dividendos pagados ese día (si existen).
- STOCK_SPLITS: Split de acciones ocurrido ese día (si existe).
- SMA_5: Media móvil simple del cierre en los últimos 5 días.
- SMA_10: Media móvil simple del cierre en los últimos 10 días.
- SMA_20: Media móvil simple del cierre en los últimos 20 días.
- SMA_50: Media móvil simple del cierre en los últimos 50 días.
- EMA_12: Media móvil exponencial de 12 días.
- EMA_26: Media móvil exponencial de 26 días.
- MACD: Diferencia entre EMA_12 y EMA_26 (indicador de momentum).
- MACD_SIGNAL: Media móvil del MACD.
- MACD_HISTOGRAM: Diferencia entre MACD y su señal.
- RSI: Índice de Fuerza Relativa (14 periodos).
- VOLATILITY: Volatilidad histórica del precio.
- BB_MIDDLE: Banda central de Bollinger (media móvil).
- BB_UPPER: Banda superior de Bollinger.
- BB_LOWER: Banda inferior de Bollinger.
- BB_WIDTH: Ancho de las bandas de Bollinger (medida de volatilidad relativa).
- BB_POSITION: Posición del precio dentro de las bandas de Bollinger.
- PRICE_CHANGE: Cambio diario del precio de cierre.
- PRICE_CHANGE_5D: Cambio acumulado del precio en los últimos 5 días.
- HIGH_LOW_RATIO: Relación entre precio máximo y mínimo.
- OPEN_CLOSE_RATIO: Relación entre apertura y cierre.
- VOLUME_SMA: Media móvil del volumen.
- VOLUME_RATIO: Volumen actual comparado con su media.
- CLOSE_LAG_1: Precio de cierre de hace 1 día.
- CLOSE_LAG_2: Precio de cierre de hace 2 días.
- CLOSE_LAG_3: Precio de cierre de hace 3 días.
- CLOSE_LAG_4: Precio de cierre de hace 4 días.
- CLOSE_LAG_5: Precio de cierre de hace 5 días.
- CLOSE_LAG_6: Precio de cierre de hace 6 días.
- CLOSE_LAG_7: Precio de cierre de hace 7 días.
- CLOSE_LAG_8: Precio de cierre de hace 8 días.
- CLOSE_LAG_9: Precio de cierre de hace 9 días.
- CLOSE_LAG_10: Precio de cierre de hace 10 días.
- VOLUME_LAG_1: Volumen de hace 1 día.
- VOLUME_LAG_2: Volumen de hace 2 días.
- VOLUME_LAG_3: Volumen de hace 3 días.
- VOLUME_LAG_4: Volumen de hace 4 días.
- VOLUME_LAG_5: Volumen de hace 5 días.
- VOLUME_LAG_6: Volumen de hace 6 días.
- VOLUME_LAG_7: Volumen de hace 7 días.
- VOLUME_LAG_8: Volumen de hace 8 días.
- VOLUME_LAG_9: Volumen de hace 9 días.
- VOLUME_LAG_10: Volumen de hace 10 días.
- PRICE_CHANGE_LAG_1: Cambio de precio de hace 1 día.
- PRICE_CHANGE_LAG_2: Cambio de precio de hace 2 días.
- PRICE_CHANGE_LAG_3: Cambio de precio de hace 3 días.
- PRICE_CHANGE_LAG_4: Cambio de precio de hace 4 días.
- PRICE_CHANGE_LAG_5: Cambio de precio de hace 5 días.
- PRICE_CHANGE_LAG_6: Cambio de precio de hace 6 días.
- PRICE_CHANGE_LAG_7: Cambio de precio de hace 7 días.
- PRICE_CHANGE_LAG_8: Cambio de precio de hace 8 días.
- PRICE_CHANGE_LAG_9: Cambio de precio de hace 9 días.
- PRICE_CHANGE_LAG_10: Cambio de precio de hace 10 días.
- RSI_LAG_1: RSI de hace 1 día.
- RSI_LAG_2: RSI de hace 2 días.
- RSI_LAG_3: RSI de hace 3 días.
- RSI_LAG_4: RSI de hace 4 días.
- RSI_LAG_5: RSI de hace 5 días.
- MACD_LAG_1: MACD de hace 1 día.
- MACD_LAG_2: MACD de hace 2 días.
- MACD_LAG_3: MACD de hace 3 días.
- MACD_LAG_4: MACD de hace 4 días.
- MACD_LAG_5: MACD de hace 5 días.
- VOLATILITY_LAG_1: Volatilidad de hace 1 día.
- VOLATILITY_LAG_2: Volatilidad de hace 2 días.
- FUTURE_RETURN_1D / 5D / 10D / 20D: Retorno futuro a N días.
- FUTURE_UP_1D / 5D / 10D / 20D: Variable binaria (1 si sube, 0 si no).
- FUTURE_CATEGORY_1D / 5D / 10D / 20D: Categoría del retorno futuro.

## Cargar datos y crear DataFrame

In [ ]:
# Definir ruta donde estan los archivos
path = r'../data/raw/'

In [ ]:
# Listar archivos Parquet que comienzan con 'data'
all_files = glob.glob(os.path.join(path, 'data*.parquet'))

In [ ]:
# Leer cada archivo Parquet y almacenarlo en una lista de DataFrames
df_list = []
for filename in all_files:
    df_part = pd.read_parquet(filename, engine='pyarrow')
    df_list.append(df_part)

In [ ]:
# Concatenar todos los DataFrames en un único DataFrame
df = pd.concat(df_list, axis=0, ignore_index=True)

In [ ]:
# Convertimos Date a datetime (necesario para split temporal)
df['Date'] = pd.to_datetime(df['Date'])

## Análisis descriptivo

In [ ]:
# Mostrar las primeras filas del dataset
df.head()

In [ ]:
# Ver el número de filas y columnas
df.shape

In [ ]:
# Mostrar los nombres de las columnas del DataFrame
df.columns

In [ ]:
# Estadísticas descriptivas de las variables numéricas
df.describe()

In [ ]:
# Información general del DataFrame
df.info()

In [ ]:
# Contar valores nulos por columna
df.isnull().sum().sort_values(ascending=False)

### Observaciones
- El dataset tiene un total de 620,095 filas.
- Cuenta con 73 columnas en total.
- No se han encontrado filas duplicadas en el dataset.
- No hay valores nulos en ninguna columna.
- De las 73 columnas, 71 son de tipo numérico.
- Dos columnas son de tipo objeto.

## Limpieza de Datos

In [ ]:
# Ordenamos por Ticker y Date
df = df.sort_values(['Ticker','Date'])

In [ ]:
# Eliminamos columnas que no serán utilizadas inicialmente para el análisis exploratorio ni para el modelado.
df = df.drop(['Dividends', 'Stock Splits', 'SMA_10', 'SMA_50', 'MACD_Histogram', 'BB_Middle',
              'BB_Upper', 'BB_Lower', 'BB_Width', 'Price_Change_5d', 'High_Low_Ratio', 'Open_Close_Ratio', 'Volume_SMA',
              'Close_lag_5', 'Close_lag_10','Price_Change_lag_1', 'Price_Change_lag_2',
              'Price_Change_lag_3', 'Price_Change_lag_5', 'Price_Change_lag_10', 'RSI_lag_2', 'RSI_lag_3', 'RSI_lag_5', 'RSI_lag_10',
              'MACD_lag_2', 'MACD_lag_3', 'MACD_lag_5', 'MACD_lag_10', 'Future_Category_1d', 'Future_Return_5d',
              'Future_Up_5d', 'Future_Category_5d', 'Future_Return_10d','Future_Up_10d', 'Future_Category_10d', 
              'Future_Return_20d', 'Future_Up_20d', 'Future_Category_20d','Future_Return_1d'], axis=1)

### Observaciones
Se eliminaron las columnas que no aportan información relevante para el objetivo principal del análisis y modelado, como métricas a largo plazo, indicadores redundantes o futuros retornos de 5, 10 y 20 días, para simplificar el dataset y evitar data leakage. Se eligió Future_Up_1d como variable objetivo porque representa de manera binaria si la acción subirá al menos un 1 % al día siguiente, alineándose directamente con el objetivo del modelo.

In [ ]:
# Contar cuántos valores cero hay en cada columna del DataFrame.
value_cero = (df == 0).sum()

In [ ]:
# Mostrar solo las columnas que contienen al menos un valor cero
value_cero[value_cero > 0]

In [ ]:
# Revisar la distribución de los valores de la columna 'Price_Change'
df['Price_Change'].value_counts()

### Observaciones
Se comprueba la cantidad de ceros en el dataset para asegurarnos de que no representan errores o datos faltantes. Aunque Price_Change tiene varios ceros, esto es esperado y consistente con los movimientos reales de precio; por lo tanto, no se consideran anomalías.

## Visualización

In [ ]:
# Creamos un DataFrame solo con columnas numéricas (excluyendo 'Ticker' y 'Date')
df_filtrado = df.drop(columns=['Ticker','Date'])

# Calculamos la matriz de correlación entre las variables numéricas
corr_numerical = df_filtrado.corr(numeric_only=True)

# Filtramos correlaciones fuertes y preparamos la máscara para la mitad superior
high_correlation = corr_numerical[corr_numerical.abs() > 0.40]
mask = np.triu(np.ones_like(corr_numerical, dtype=bool))

# Graficamos el mapa de calor de correlaciones
fig, axis = plt.subplots(figsize=(15,15))
sns.heatmap(high_correlation, annot=True, mask=mask, linewidths=3, fmt='.2f')

plt.tight_layout()
plt.show()

In [ ]:
# Eliminamos columnas altamente redundantes detectadas en el mapa de calor de correlaciones
cols_to_drop = ['Open', 'High', 'Low', 'SMA_5', 'SMA_20', 'EMA_26', 'Close_lag_1', 'Close_lag_2',
                'Close_lag_3', 'Volatility_lag_2', 'Volatility_lag_3', 'Volatility_lag_5', 'Volatility_lag_10']

df = df.drop(columns=cols_to_drop)

### Observaciones
Se eliminaron estas columnas porque presentaban correlaciones muy altas (0.85 o superiores) con otras variables, lo que indica redundancia y riesgo de multicolinealidad. Ninguna de ellas mostraba relación directa con el target Future_Up_1d, por lo que su eliminación permite simplificar el dataset sin perder información relevante para el modelado.

In [ ]:
# Verificamos cómo se relaciona el target 'Future_Up_1d' con el cambio de precio al día siguiente
verificacion = df[['Ticker', 'Price_Change', 'Future_Up_1d','Date']].copy()
verificacion['Next_Day_Change'] = verificacion.groupby('Ticker')['Price_Change'].shift(-1)

In [ ]:
# Calculamos el cambio promedio cuando el target es 1
verificacion[verificacion['Future_Up_1d'] == 1]['Next_Day_Change'].mean()

### Conclusión
Los resultados muestran que el cambio promedio cuando Future_Up_1d es 1 es de aproximadamente 0.014 (1.41%). Esto confirma que nuestra variable objetivo captura correctamente los movimientos significativos al alza que queremos predecir.

## Split Temporal

In [ ]:
# Definimos fecha de corte (80% más antiguo para train)
split_date = df['Date'].quantile(0.8)

In [ ]:
# Creamos datasets temporales
train_df = df[df['Date'] <= split_date]
test_df  = df[df['Date'] > split_date]

In [ ]:
# Definimos X, y, y eliminamos Date (no entra en nuestro modelo)
X_train = train_df.drop(columns=['Future_Up_1d','Date'])
y_train = train_df['Future_Up_1d']

X_test = test_df.drop(columns=['Future_Up_1d','Date'])
y_test = test_df['Future_Up_1d']

## Target Encoding

Se utilizó Target Encoding para la columna Ticker porque es una variable categórica de alta cardinalidad (más de 500 categorías). Aplicar One-Hot Encoding habría generado cientos de columnas adicionales, aumentando innecesariamente la dimensionalidad del dataset, el costo computacional y el riesgo de sobreajuste. Target Encoding permite representar cada Ticker con un único valor numérico basado en su relación con la variable objetivo, manteniendo la información relevante sin expandir el espacio de características. Esto hace el modelo más eficiente, estable y escalable.

In [ ]:
# Separamos las variables numéricas de la variable categórica 'Ticker'
X_train_num = X_train.drop(columns=['Ticker'])
X_test_num = X_test.drop(columns=['Ticker'])

# Aplicamos Target Encoding a la variable categórica
target_encoder = TargetEncoder(cols=['Ticker'])

# Ajustamos el encoder únicamente con los datos de entrenamiento
X_train_ticker_encoded = target_encoder.fit_transform(X_train[['Ticker']], y_train)

# Transformamos el conjunto de test usando el encoder ya entrenado
X_test_ticker_encoded = target_encoder.transform(X_test[['Ticker']])

# Unimos las variables numéricas originales con el ticker codificado
X_train_final = pd.concat([X_train_num, X_train_ticker_encoded], axis=1)
X_test_final = pd.concat([X_test_num, X_test_ticker_encoded], axis=1)

## Standard Scaler

In [ ]:
# Instanciamos el Scaler
scaler = StandardScaler()

# Ajustamos con los datos completos (numéricos + ticker encoded)
scaler.fit(X_train_final)

# Transformamos ambos conjuntos
X_train_scaled_array = scaler.transform(X_train_final)
X_test_scaled_array  = scaler.transform(X_test_final)

# Convertimos de nuevo a DataFrame para mantener el orden y nombres de columnas
X_train_scaled = pd.DataFrame(X_train_scaled_array, columns=X_train_final.columns, index=X_train_final.index)
X_test_scaled = pd.DataFrame(X_test_scaled_array, columns=X_test_final.columns, index=X_test_final.index)

## Guardar datos de entrenamiento y prueba

In [ ]:
# Guardar los conjuntos de prueba y entrenamiento procesados como archivos CSV
X_train_scaled.to_csv("../data/processed/X_train_temporal_obj1.csv", index = False)
X_test_scaled.to_csv("../data/processed/X_test_temporal_obj1.csv", index = False)
y_train.to_csv("../data/processed/y_train_temporal_obj1.csv", index = False)
y_test.to_csv("../data/processed/y_test_temporal_obj1.csv", index = False)